# 11.15 — Actor-Critic

Actor-critic methods learn two things at once: an **actor** policy that chooses actions and a **critic** value function that estimates how good the current state is. The critic turns delayed reward into a lower-variance advantage signal, so the actor can reinforce actions that did better than expected and weaken actions that did worse.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build actor-critic one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is visible, from discounted return to advantage to the actor and critic updates. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random choices, and small table updates.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for policy sampling and toy training.

### 1. Consequence means discounted return, not immediate reward

Reinforcement learning starts with a time sequence. An action gives an immediate reward, but it also changes which future states and rewards become possible. The discounted return $$G_t=r_t+\gamma r_{t+1}+\gamma^2 r_{t+2}+\cdots$$ is the ledger that adds those future consequences while making farther rewards count a little less.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # reward now, next step, and two steps later.
gamma_w = 0.9  # discount factor: future rewards still matter, but less than immediate reward.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2].
terms_w = powers_w * rewards_w  # each reward's discounted contribution.
print("discount powers:", np.round(powers_w, 3))
print("discounted terms:", np.round(terms_w, 3))

▶ What you'll see: the delayed reward of 2 contributes `0.9²·2 = 1.62`, not the full 2.

In [ ]:
G_w = float(np.sum(terms_w))  # total discounted return.
print("three-step return:", round(G_w, 3))
assert round(G_w, 3) == 2.620  # concrete number from the lesson content.
plt.figure(figsize=(4.4, 3))
plt.bar(["r0", "γr1", "γ²r2"], terms_w, color="teal")
plt.title("1: discounted pieces of return")
plt.ylabel("contribution to G")
plt.show()

▶ What you'll see: most return comes from the immediate `1` plus the delayed discounted `1.62`.

*Why it's done this way:* the actor should not chase immediate reward alone, because many useful actions pay off later. Discounting is a controlled compromise: $\gamma<1$ keeps infinite or distant sums bounded and expresses that sooner evidence is usually more reliable than far-future evidence.

### 2. A tiny environment where actions create future evidence

We will use a two-state toy environment. State 0 is a decision state: action 0 gives a small safe reward and ends; action 1 gives no immediate reward but moves to state 1. In state 1, either action ends with reward 2. This is the smallest setting where an immediate-reward rule prefers action 0, while a consequence-aware rule can prefer action 1.

In [ ]:
def step_env_w(state_w, action_w):  # deterministic toy environment.
    if state_w == 0 and action_w == 0:
        return 0, 1.0, True  # safe: immediate reward, episode ends.
    if state_w == 0 and action_w == 1:
        return 1, 0.0, False  # invest: no reward now, move to payoff state.
    return 1, 2.0, True  # payoff state: terminal reward.

for action_w in [0, 1]:
    print("from state 0, action", action_w, "->", step_env_w(0, action_w))

▶ What you'll see: action 0 pays `1` now; action 1 pays `0` now but reaches state 1.

In [ ]:
safe_return_w = 1.0  # action 0 ends immediately.
invest_return_w = 0.0 + gamma_w * 2.0  # action 1 reaches a later reward of 2.
print("safe return:", safe_return_w)
print("invest return:", invest_return_w)
assert round(invest_return_w, 3) == 1.800
plt.figure(figsize=(4, 3))
plt.bar(["safe a0", "invest a1"], [safe_return_w, invest_return_w], color=["gray", "seagreen"])
plt.title("2: immediate reward can pick the wrong action")
plt.ylabel("discounted return")
plt.show()

▶ What you'll see: the action with lower immediate reward has higher discounted return.

*Why it's done this way:* actor-critic needs an environment loop because the policy affects its own future data. This toy strips away distractions so the central RL issue is visible: an action is good when its future consequence is good, not merely when its current reward is high.

### 3. The actor is a softmax policy

The actor stores logits, one score per action in each state. Softmax converts logits into probabilities: $$\pi(a\mid s)=\frac{e^{z_a}}{\sum_b e^{z_b}}.$$ A logit is not itself a probability; it becomes a probability only after comparing it to the other action logits in the same state.

In [ ]:
def softmax_w(logits_w):
    shifted_w = logits_w - np.max(logits_w)  # numerical stability without changing probabilities.
    exp_w = np.exp(shifted_w)
    return exp_w / np.sum(exp_w)

logits_w = np.array([1.0, 0.0])
probs_w = softmax_w(logits_w)
print("softmax probabilities:", np.round(probs_w, 3))
assert np.allclose(np.round(probs_w, 3), [0.731, 0.269])

▶ What you'll see: logit advantage of 1 turns into probabilities about `0.731` and `0.269`.

In [ ]:
rewards_actions_w = np.array([2.0, 0.0])
expected_reward_w = float(np.dot(probs_w, rewards_actions_w))
print("expected one-step reward:", round(expected_reward_w, 3))
assert round(expected_reward_w, 3) == 1.462
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], probs_w, color="purple")
plt.title("3: actor probabilities from logits")
plt.ylabel("π(a|s)")
plt.show()

▶ What you'll see: most probability mass goes to action 0, so the expected reward is mostly action 0's reward.

*Why it's done this way:* softmax keeps exploration alive because every action has nonzero probability, while still making better logits more likely. The actor update will move logits, and softmax translates those moves into probability mass and therefore expected consequence.

### 4. The critic learns a bootstrapped value target

The critic estimates $$V(s)$$, the expected discounted return from a state under the current policy. Instead of waiting for a full episode every time, it can bootstrap with a one-step target $$y=r+\gamma V(s')$$. The temporal-difference error $$\delta=y-V(s)$$ says whether the outcome was better or worse than the critic expected.

In [ ]:
V_w = np.array([0.4, 0.8])  # current critic estimates for states 0 and 1.
r_w = 1.0
next_state_w = 1
target_w = r_w + gamma_w * V_w[next_state_w]
td_error_w = target_w - V_w[0]
print("target y:", round(target_w, 3))
print("TD error δ:", round(td_error_w, 3))
assert round(target_w, 3) == 1.720

▶ What you'll see: the target is `1 + 0.9·0.8 = 1.72`, above the old value `0.4`.

In [ ]:
alpha_v_w = 0.5
V_new0_w = V_w[0] + alpha_v_w * td_error_w
print("updated V(0):", round(V_new0_w, 3))
assert round(V_new0_w, 3) == 1.060
plt.figure(figsize=(4, 3))
plt.bar(["old V(0)", "target y", "new V(0)"], [V_w[0], target_w, V_new0_w], color=["gray", "black", "teal"])
plt.title("4: critic moves partway toward target")
plt.ylabel("value")
plt.show()

▶ What you'll see: the critic moves halfway from `0.4` toward `1.72`.

*Why it's done this way:* bootstrapping lowers variance because the critic does not wait for every future reward, but it adds bias because the target contains the critic's own estimate. The learning rate controls how much one noisy transition can change the value table.

### 5. Advantage tells the actor what was better than expected

The actor should not reinforce every rewarded action equally. If a state was already expected to be good, a reward may be ordinary; if a state was expected to be bad, the same reward may be surprising. Actor-critic uses the TD error as a one-step advantage estimate: $$A_t\approx\delta_t=r_t+\gamma V(s_{t+1})-V(s_t).$$

In [ ]:
V_adv_w = np.array([1.0, 2.0])  # critic expects state 0 to be worth 1 and state 1 to be worth 2.
next_good_w = 1
adv_invest_w = 0.0 + gamma_w * V_adv_w[next_good_w] - V_adv_w[0]
adv_safe_w = 1.0 + 0.0 - V_adv_w[0]
print("advantage safe:", round(adv_safe_w, 3))
print("advantage invest:", round(adv_invest_w, 3))
assert round(adv_invest_w, 3) == 0.800

▶ What you'll see: the safe action is exactly as expected (`0` advantage), while investing is better than expected (`0.8`).

In [ ]:
advantages_w = np.array([adv_safe_w, adv_invest_w])
plt.figure(figsize=(4, 3))
plt.bar(["safe a0", "invest a1"], advantages_w, color=["gray", "seagreen"])
plt.axhline(0, color="black", linewidth=1)
plt.title("5: advantage is reward relative to expectation")
plt.ylabel("A ≈ δ")
plt.show()

▶ What you'll see: positive advantage marks the action whose consequence beat the critic's baseline.

*Why it's done this way:* subtracting a baseline does not change the expected policy-gradient direction, but it reduces variance. The actor now learns from “better or worse than expected,” not raw return magnitude, so common state quality is absorbed by the critic rather than incorrectly credited to every sampled action.

### 6. The actor update moves probability toward positive advantage

For a softmax actor, the gradient of log probability for the chosen action has a simple shape: selected action gets `1 - p_selected`, and unselected actions get `-p_other`. Multiplying by advantage increases the chosen action's logit if the advantage is positive and decreases it if the advantage is negative.

In [ ]:
logits_actor_w = np.array([0.2, -0.2])
probs_actor_w = softmax_w(logits_actor_w)
a_w = 1  # suppose the actor chose invest.
A_w = 0.8  # the critic says it was better than expected.
grad_logp_w = -probs_actor_w.copy()
grad_logp_w[a_w] += 1.0
print("policy probs before:", np.round(probs_actor_w, 3))
print("grad log π(a|s):", np.round(grad_logp_w, 3))

▶ What you'll see: the chosen action gets a positive gradient component and the other action gets a negative one.

In [ ]:
alpha_pi_w = 0.4
new_logits_actor_w = logits_actor_w + alpha_pi_w * A_w * grad_logp_w
new_probs_actor_w = softmax_w(new_logits_actor_w)
print("policy probs after:", np.round(new_probs_actor_w, 3))
assert new_probs_actor_w[1] > probs_actor_w[1]
plt.figure(figsize=(4, 3))
plt.bar(["before invest", "after invest"], [probs_actor_w[1], new_probs_actor_w[1]], color=["gray", "seagreen"])
plt.title("6: positive advantage raises chosen probability")
plt.ylabel("π(invest|state 0)")
plt.show()

▶ What you'll see: probability of the chosen invest action rises after the positive-advantage update.

*Why it's done this way:* the policy-gradient theorem says to move parameters in proportion to $\nabla\log\pi(a_t\mid s_t)A_t$. The log-probability gradient says how to make the sampled action more or less likely; the advantage says whether that sampled action deserves reinforcement or punishment.

### 7. Training actor and critic together

Now we combine the pieces: sample an action from the actor, step the environment, compute the critic's TD error, update the critic toward the bootstrap target, and update the actor with the same TD error as advantage. Even in this tiny table version, that is the actor-critic loop.

In [ ]:
logits_train_w = np.zeros((2, 2))  # two states, two action logits per state.
V_train_w = np.zeros(2)  # critic value table.
prob_history_w = []
value_history_w = []
rng_train_w = np.random.default_rng(0)
print("initial π(state0):", np.round(softmax_w(logits_train_w[0]), 3))
print("initial V:", V_train_w)

▶ What you'll see: the actor starts uniform and the critic starts at zero.

In [ ]:
for episode_w in range(160):
    state_w = 0
    done_w = False
    while not done_w:
        probs_step_w = softmax_w(logits_train_w[state_w])
        action_w = int(rng_train_w.choice(2, p=probs_step_w))
        next_state_w, reward_w, done_w = step_env_w(state_w, action_w)
        target_step_w = reward_w + (0.0 if done_w else gamma_w * V_train_w[next_state_w])
        delta_step_w = target_step_w - V_train_w[state_w]
        V_train_w[state_w] += 0.2 * delta_step_w
        grad_step_w = -probs_step_w
        grad_step_w[action_w] += 1.0
        logits_train_w[state_w] += 0.15 * delta_step_w * grad_step_w
        state_w = next_state_w
    prob_history_w.append(softmax_w(logits_train_w[0])[1])
    value_history_w.append(V_train_w[0])
print("final π(invest|state0):", round(prob_history_w[-1], 3))
print("final V(0):", round(value_history_w[-1], 3))
assert prob_history_w[-1] > 0.75

▶ What you'll see: the actor learns to favor the delayed-payoff action and the critic's value rises.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(prob_history_w, label="π(invest|s0)", color="seagreen")
plt.plot(value_history_w, label="V(s0)", color="purple")
plt.title("7: actor and critic improve together")
plt.xlabel("episode")
plt.legend()
plt.show()

▶ What you'll see: probability and value climb together, with small sampling wiggles.

*Why it's done this way:* the critic supplies a lower-variance learning signal immediately after each transition, while the actor changes the data distribution by choosing better actions more often. The two estimates are coupled: a better critic gives cleaner advantages, and a better actor visits better trajectories.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for tables, softmax, sampling, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for the small diagnostic plots.
np.random.seed(0) # make random examples reproducible across notebook runs.

def softmax(x): # convert logits to probabilities in a numerically stable way.
    x = np.asarray(x, dtype=float) # ensure array arithmetic.
    z = x - np.max(x) # subtract max without changing softmax probabilities.
    e = np.exp(z) # exponentiate shifted logits.
    return e / np.sum(e) # normalize into probabilities that sum to 1.

def toy_step(state, action): # tiny deterministic environment used by the examples.
    if state == 0 and action == 0: # safe action from the start state.
        return 0, 1.0, True # immediate reward and terminal.
    if state == 0 and action == 1: # investment action from the start state.
        return 1, 0.0, False # no reward now, but transition to payoff state.
    return 1, 2.0, True # payoff state always ends with reward 2.

def discounted_return(rewards, gamma): # compute G from a finite reward list.
    rewards = np.asarray(rewards, dtype=float) # make powers and rewards broadcast cleanly.
    return float(np.sum((gamma ** np.arange(len(rewards))) * rewards)) # sum γ^t r_t.

def grad_log_softmax(probs, action): # gradient of log π(action) with respect to logits.
    g = -np.asarray(probs, dtype=float).copy() # unchosen actions get -probability.
    g[int(action)] += 1.0 # chosen action gets 1 - probability.
    return g # return a vector with the same length as the action space.

## 🟢 Basics (warm-up)

### Basic 1 — Compute a discounted return

**Goal.** Turn a short reward stream into one consequence number, because actor-critic optimizes return rather than only the first reward. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0]) # define rewards at t, t+1, and t+2.
gamma_b1 = 0.9 # choose the lesson discount factor.
terms_b1 = (gamma_b1 ** np.arange(len(rewards_b1))) * rewards_b1 # compute each discounted contribution.
print("discounted terms:", np.round(terms_b1, 3)) # inspect the pieces before summing.

▶ What you'll see: the third reward contributes `1.62` after two discounts.

In [ ]:
G_b1 = discounted_return(rewards_b1, gamma_b1) # sum the discounted reward stream.
print("G:", round(G_b1, 3)) # inspect the return.
assert round(G_b1, 3) == 2.620 # verify the concrete worked number.
plt.figure(figsize=(4, 3)) # create a compact contribution plot.
plt.bar(["t", "t+1", "t+2"], terms_b1, color="teal") # show each discounted term.
plt.title("Basic 1: discounted return pieces") # title the plot.
plt.ylabel("γ^k r") # label the contribution scale.
plt.show() # display the plot.

▶ What you'll see: return is the sum of current and discounted future reward contributions.

👀 Takeaway: reward is local, while return is the action's delayed consequence.

### Basic 2 — Compare immediate reward and return

**Goal.** Show why a delayed-payoff action can be better than a greedy action, because actor-critic exists to solve delayed-credit problems. We build it in 2 steps.

In [ ]:
next_safe_b2, r_safe_b2, done_safe_b2 = toy_step(0, 0) # take the safe action from state 0.
next_inv_b2, r_inv_b2, done_inv_b2 = toy_step(0, 1) # take the investment action from state 0.
print("safe transition:", (next_safe_b2, r_safe_b2, done_safe_b2)) # inspect safe outcome.
print("invest transition:", (next_inv_b2, r_inv_b2, done_inv_b2)) # inspect investment outcome.

▶ What you'll see: safe pays immediately; invest moves to the payoff state.

In [ ]:
returns_b2 = np.array([r_safe_b2, r_inv_b2 + 0.9 * 2.0]) # compute consequence of each first action.
print("first-action returns:", returns_b2) # inspect safe versus investment return.
assert round(float(returns_b2[1]), 3) == 1.800 # verify delayed-payoff return.
plt.figure(figsize=(4, 3)) # create a compact comparison plot.
plt.bar(["safe", "invest"], returns_b2, color=["gray", "seagreen"]) # compare returns.
plt.title("Basic 2: delayed payoff beats greedy reward") # title the plot.
plt.ylabel("return from state 0") # label the return scale.
plt.show() # display the plot.

▶ What you'll see: the lower-immediate-reward action has the larger discounted return.

👀 Takeaway: RL actions must be judged by the future they unlock.

### Basic 3 — Convert logits to action probabilities

**Goal.** Build the actor's softmax probabilities from logits, because the policy must choose actions stochastically. We build it in 2 steps.

In [ ]:
logits_b3 = np.array([1.0, 0.0]) # define two action logits.
exp_b3 = np.exp(logits_b3 - np.max(logits_b3)) # exponentiate shifted logits for stability.
print("shifted exponentials:", np.round(exp_b3, 3)) # inspect softmax numerator pieces.

▶ What you'll see: the larger logit has the larger exponential weight.

In [ ]:
probs_b3 = softmax(logits_b3) # convert logits to probabilities.
print("probabilities:", np.round(probs_b3, 3), "sum:", round(float(np.sum(probs_b3)), 3)) # inspect normalized probabilities.
assert np.allclose(np.round(probs_b3, 3), [0.731, 0.269]) # verify lesson probabilities.
plt.figure(figsize=(4, 3)) # create a policy bar chart.
plt.bar(["a0", "a1"], probs_b3, color="purple") # show probability mass by action.
plt.title("Basic 3: softmax policy") # title the plot.
plt.ylabel("π(a|s)") # label the policy scale.
plt.show() # display the plot.

▶ What you'll see: probabilities are positive and sum to 1.

👀 Takeaway: actor logits become a probability distribution only after softmax normalization.

### Basic 4 — Compute expected reward under a policy

**Goal.** Weight action rewards by policy probabilities, because a stochastic policy is evaluated by expectation. We build it in 2 steps.

In [ ]:
probs_b4 = softmax(np.array([1.0, 0.0])) # define the actor's action probabilities.
rewards_b4 = np.array([2.0, 0.0]) # define one-step rewards for the two actions.
weighted_b4 = probs_b4 * rewards_b4 # compute each action's contribution to expectation.
print("weighted rewards:", np.round(weighted_b4, 3)) # inspect contribution by action.

▶ What you'll see: action 0 dominates the expected reward because it has high probability and reward.

In [ ]:
expected_b4 = float(np.sum(weighted_b4)) # sum probability-weighted rewards.
print("expected reward:", round(expected_b4, 3)) # inspect the expectation.
assert round(expected_b4, 3) == 1.462 # verify the lesson number.
plt.figure(figsize=(4, 3)) # create a contribution chart.
plt.bar(["a0", "a1"], weighted_b4, color="teal") # show expected-reward pieces.
plt.title("Basic 4: probability-weighted reward") # title the plot.
plt.ylabel("π(a) r(a)") # label the contribution scale.
plt.show() # display the plot.

▶ What you'll see: expected reward is the sum of the weighted bars.

👀 Takeaway: changing actor probabilities changes expected consequence.

### Basic 5 — Build a one-step critic target

**Goal.** Compute `r + γV(next)`, because the critic improves by bootstrapping from its next-state estimate. We build it in 2 steps.

In [ ]:
V_b5 = np.array([0.4, 0.8]) # current critic estimates for two states.
r_b5 = 1.0 # observed reward.
gamma_b5 = 0.9 # discount factor.
y_b5 = r_b5 + gamma_b5 * V_b5[1] # one-step bootstrap target.
print("target y:", round(y_b5, 3)) # inspect target value.
assert round(y_b5, 3) == 1.720 # verify lesson target.

▶ What you'll see: the next-state value contributes `0.72` to the target.

In [ ]:
plt.figure(figsize=(4, 3)) # create a target decomposition plot.
plt.bar(["reward r", "γV(next)", "target y"], [r_b5, gamma_b5 * V_b5[1], y_b5], color=["gray", "teal", "black"]) # show target pieces.
plt.title("Basic 5: bootstrap target") # title the plot.
plt.ylabel("value") # label the value scale.
plt.xticks(rotation=15) # rotate labels to fit.
plt.show() # display the plot.

▶ What you'll see: the target combines immediate reward and discounted estimated future value.

👀 Takeaway: bootstrapping replaces a long unknown future with the critic's current estimate.

### Basic 6 — Update the critic toward the target

**Goal.** Move a value estimate partway toward the one-step target, because learning rates prevent one transition from overwriting the critic. We build it in 2 steps.

In [ ]:
V_old_b6 = 0.4 # old value for the current state.
target_b6 = 1.72 # bootstrap target from Basic 5.
alpha_b6 = 0.5 # critic learning rate.
delta_b6 = target_b6 - V_old_b6 # temporal-difference error.
print("TD error:", round(delta_b6, 3)) # inspect the correction signal.

▶ What you'll see: positive TD error means the outcome was better than the critic expected.

In [ ]:
V_new_b6 = V_old_b6 + alpha_b6 * delta_b6 # move halfway toward the target.
print("new value:", round(V_new_b6, 3)) # inspect the updated estimate.
assert round(V_new_b6, 3) == 1.060 # verify the worked update.
plt.figure(figsize=(4, 3)) # create a before-target-after plot.
plt.bar(["old", "target", "new"], [V_old_b6, target_b6, V_new_b6], color=["gray", "black", "teal"]) # show the partial move.
plt.title("Basic 6: critic update") # title the plot.
plt.ylabel("V(s)") # label the value scale.
plt.show() # display the plot.

▶ What you'll see: the new value lands exactly halfway between old value and target.

👀 Takeaway: TD learning is a controlled correction toward a bootstrapped target.

### Basic 7 — Interpret TD error as advantage

**Goal.** Treat TD error as a one-step advantage estimate, because the actor needs to know whether the sampled action beat expectation. We build it in 2 steps.

In [ ]:
V_b7 = np.array([1.0, 2.0]) # critic baseline values.
gamma_b7 = 0.9 # discount factor.
adv_safe_b7 = 1.0 - V_b7[0] # safe action ends, so no next value.
adv_invest_b7 = 0.0 + gamma_b7 * V_b7[1] - V_b7[0] # invest action bootstraps from state 1.
print("advantages:", np.round([adv_safe_b7, adv_invest_b7], 3)) # inspect baseline-relative outcomes.

▶ What you'll see: safe is ordinary and invest is better than expected.

In [ ]:
assert round(adv_invest_b7, 3) == 0.800 # verify the positive advantage.
plt.figure(figsize=(4, 3)) # create an advantage plot.
plt.bar(["safe", "invest"], [adv_safe_b7, adv_invest_b7], color=["gray", "seagreen"]) # compare action advantages.
plt.axhline(0, color="black", linewidth=1) # add zero baseline.
plt.title("Basic 7: advantage = target - baseline") # title the plot.
plt.ylabel("A") # label the advantage scale.
plt.show() # display the plot.

▶ What you'll see: only actions above the critic baseline get positive reinforcement.

👀 Takeaway: advantage assigns credit relative to what the critic already expected from the state.

### Basic 8 — Compute the log-policy gradient

**Goal.** Find how logits should change to make a sampled action more likely, because the actor update uses `∇ log π(a|s)`. We build it in 2 steps.

In [ ]:
probs_b8 = softmax(np.array([0.2, -0.2])) # current policy probabilities.
a_b8 = 1 # sampled action.
grad_b8 = grad_log_softmax(probs_b8, a_b8) # gradient of log probability for sampled action.
print("probs:", np.round(probs_b8, 3)) # inspect policy before gradient.
print("grad log-prob:", np.round(grad_b8, 3)) # inspect gradient vector.

▶ What you'll see: the chosen action's component is positive and the unchosen component is negative.

In [ ]:
print("gradient sums to:", round(float(np.sum(grad_b8)), 6)) # softmax logit gradients sum to zero.
assert abs(float(np.sum(grad_b8))) < 1e-12 # verify probability mass is redistributed, not created.
plt.figure(figsize=(4, 3)) # create a gradient bar chart.
plt.bar(["logit a0", "logit a1"], grad_b8, color=["crimson", "seagreen"]) # show logit update directions.
plt.axhline(0, color="black", linewidth=1) # add zero line.
plt.title("Basic 8: ∇ log π for chosen action") # title the plot.
plt.show() # display the plot.

▶ What you'll see: increasing the chosen action's logit is balanced by decreasing the other action's relative logit.

👀 Takeaway: the softmax gradient tells the actor how to move probability mass toward or away from the sampled action.

### Basic 9 — Apply one actor update

**Goal.** Multiply the log-policy gradient by advantage, because good sampled actions should become more likely and bad sampled actions less likely. We build it in 2 steps.

In [ ]:
logits_b9 = np.array([0.2, -0.2]) # current actor logits.
probs_b9 = softmax(logits_b9) # current probabilities.
a_b9 = 1 # sampled action.
A_b9 = 0.8 # positive advantage.
step_b9 = 0.4 * A_b9 * grad_log_softmax(probs_b9, a_b9) # actor logit step.
print("logit step:", np.round(step_b9, 3)) # inspect how each logit changes.

▶ What you'll see: the selected action's logit increases while the other decreases.

In [ ]:
new_logits_b9 = logits_b9 + step_b9 # update actor logits.
new_probs_b9 = softmax(new_logits_b9) # convert updated logits to probabilities.
print("old probs:", np.round(probs_b9, 3), "new probs:", np.round(new_probs_b9, 3)) # compare policy before and after.
assert new_probs_b9[a_b9] > probs_b9[a_b9] # verify positive advantage increased chosen probability.
plt.figure(figsize=(4, 3)) # create a before-after probability plot.
plt.bar(["before", "after"], [probs_b9[a_b9], new_probs_b9[a_b9]], color=["gray", "seagreen"]) # show probability increase.
plt.title("Basic 9: actor reinforces positive advantage") # title the plot.
plt.ylabel("π(chosen action)") # label probability scale.
plt.show() # display the plot.

▶ What you'll see: probability of the chosen action rises after the update.

👀 Takeaway: the actor update is policy-gradient direction scaled by the critic's advantage estimate.

### Basic 10 — Run one actor-critic transition

**Goal.** Combine action sampling, critic update, and actor update once, because actor-critic is just these pieces in a loop. We build it in 3 steps.

In [ ]:
logits_b10 = np.zeros((2, 2)) # actor logits for two states and two actions.
V_b10 = np.zeros(2) # critic value table.
state_b10 = 0 # start in state 0.
a_b10 = 1 # choose the investment action for this inspected transition.
probs_b10 = softmax(logits_b10[state_b10]) # policy at state 0.
next_b10, r_b10, done_b10 = toy_step(state_b10, a_b10) # step the environment.
print("transition:", (state_b10, a_b10, r_b10, next_b10, done_b10)) # inspect sampled data.

▶ What you'll see: investment moves to state 1 with no immediate reward.

In [ ]:
target_b10 = r_b10 + (0.0 if done_b10 else 0.9 * V_b10[next_b10]) # bootstrap target.
delta_b10 = target_b10 - V_b10[state_b10] # TD error / advantage.
V_b10[state_b10] += 0.2 * delta_b10 # critic update.
logits_b10[state_b10] += 0.15 * delta_b10 * grad_log_softmax(probs_b10, a_b10) # actor update.
print("target:", round(target_b10, 3), "delta:", round(delta_b10, 3)) # inspect learning signal.
print("updated V:", np.round(V_b10, 3)) # inspect critic after update.

▶ What you'll see: with zero initial values, the first investment transition has zero TD error.

In [ ]:
print("updated policy:", np.round(softmax(logits_b10[0]), 3)) # inspect actor after the transition.
assert np.allclose(softmax(logits_b10[0]), [0.5, 0.5]) # no update because delta was zero.
plt.figure(figsize=(4, 3)) # create a compact policy plot.
plt.bar(["safe", "invest"], softmax(logits_b10[0]), color="purple") # show unchanged policy.
plt.title("Basic 10: one zero-advantage transition") # title the plot.
plt.ylabel("π(a|s0)") # label probability scale.
plt.show() # display the plot.

▶ What you'll see: no advantage means no actor movement on this specific transition.

👀 Takeaway: actor-critic only changes the policy when the critic reports an outcome different from expectation.

## 🟡 Easy

### Easy 1 — Train the critic on fixed policy rollouts

**Goal.** Learn values from sampled episodes under a fixed policy, because the critic is a return predictor before it is an actor guide. We build it in 3 steps.

In [ ]:
rng_e1 = np.random.default_rng(1) # local randomness for reproducible rollouts.
V_e1 = np.zeros(2) # initialize critic values.
policy_e1 = np.array([[0.5, 0.5], [0.5, 0.5]]) # fixed exploratory policy.
print("initial V:", V_e1) # inspect critic before learning.

▶ What you'll see: both state values start at zero.

In [ ]:
for episode_e1 in range(120): # collect many short episodes.
    state_e1 = 0 # reset environment.
    done_e1 = False # episode flag.
    while not done_e1: # run until terminal.
        action_e1 = int(rng_e1.choice(2, p=policy_e1[state_e1])) # sample from fixed policy.
        next_e1, reward_e1, done_e1 = toy_step(state_e1, action_e1) # environment transition.
        target_e1 = reward_e1 + (0.0 if done_e1 else 0.9 * V_e1[next_e1]) # one-step TD target.
        V_e1[state_e1] += 0.15 * (target_e1 - V_e1[state_e1]) # critic update.
        state_e1 = next_e1 # move forward.
print("learned V:", np.round(V_e1, 3)) # inspect learned values.
assert V_e1[0] > 1.0 # fixed policy has positive value at the start state.

▶ What you'll see: the critic learns that the start state is worth more than the immediate safe reward alone.

In [ ]:
plt.figure(figsize=(4, 3)) # create a value bar chart.
plt.bar(["V(s0)", "V(s1)"], V_e1, color="teal") # show learned state values.
plt.title("Easy 1: critic values under fixed policy") # title the plot.
plt.ylabel("value estimate") # label value scale.
plt.show() # display the plot.

▶ What you'll see: state 1 is close to its terminal payoff and state 0 blends safe and invest outcomes.

👀 Takeaway: the critic learns expected consequence for states, not which action should be selected directly.

### Easy 2 — Train actor-critic on the delayed payoff task

**Goal.** Let actor and critic learn together, because the actor needs critic advantages and the critic receives data from the actor. We build it in 4 steps.

In [ ]:
rng_e2 = np.random.default_rng(2) # local seeded generator.
logits_e2 = np.zeros((2, 2)) # actor logits.
V_e2 = np.zeros(2) # critic values.
prob_invest_e2 = [] # track π(invest|s0).
print("initial policy:", np.round(softmax(logits_e2[0]), 3)) # inspect initial actor.

▶ What you'll see: the actor starts with equal probability for safe and invest.

In [ ]:
for episode_e2 in range(220): # train over many short episodes.
    state_e2 = 0 # reset to start.
    done_e2 = False # episode flag.
    while not done_e2: # run one episode.
        probs_e2 = softmax(logits_e2[state_e2]) # actor policy at current state.
        action_e2 = int(rng_e2.choice(2, p=probs_e2)) # sample an action.
        next_e2, reward_e2, done_e2 = toy_step(state_e2, action_e2) # step environment.
        target_e2 = reward_e2 + (0.0 if done_e2 else 0.9 * V_e2[next_e2]) # critic target.
        delta_e2 = target_e2 - V_e2[state_e2] # advantage estimate.
        V_e2[state_e2] += 0.2 * delta_e2 # critic update.
        logits_e2[state_e2] += 0.12 * delta_e2 * grad_log_softmax(probs_e2, action_e2) # actor update.
        state_e2 = next_e2 # advance.
    prob_invest_e2.append(softmax(logits_e2[0])[1]) # track start-state invest probability.
print("final policy:", np.round(softmax(logits_e2[0]), 3)) # inspect learned actor.
assert prob_invest_e2[-1] > 0.75 # verify the actor learned the delayed payoff action.

▶ What you'll see: probability shifts toward the investment action.

In [ ]:
print("final critic:", np.round(V_e2, 3)) # inspect learned values.
assert V_e2[1] > 1.5 # payoff state should be valuable.

▶ What you'll see: the critic values the payoff state highly.

In [ ]:
plt.figure(figsize=(5, 3)) # create a learning curve plot.
plt.plot(prob_invest_e2, color="seagreen") # show actor improvement.
plt.title("Easy 2: actor learns delayed payoff") # title the plot.
plt.xlabel("episode") # label episode axis.
plt.ylabel("π(invest|s0)") # label policy probability.
plt.ylim(0, 1) # keep probability scale fixed.
plt.show() # display the plot.

▶ What you'll see: the policy rises toward choosing invest most of the time.

👀 Takeaway: actor-critic can prefer an action with zero immediate reward when the critic estimates its future payoff.

### Easy 3 — Compare return weighting and advantage weighting

**Goal.** Show how a critic baseline changes update variance, because subtracting expected state value removes common reward level from actor updates. We build it in 3 steps.

In [ ]:
rng_e3 = np.random.default_rng(3) # reproducible samples.
returns_e3 = rng_e3.normal(loc=5.0, scale=1.0, size=200) # noisy returns with high shared baseline.
baseline_e3 = float(np.mean(returns_e3)) # critic-like baseline estimate.
advs_e3 = returns_e3 - baseline_e3 # advantage removes common level.
print("return mean/std:", round(float(np.mean(returns_e3)), 3), round(float(np.std(returns_e3)), 3)) # inspect raw returns.
print("advantage mean/std:", round(float(np.mean(advs_e3)), 3), round(float(np.std(advs_e3)), 3)) # inspect centered advantages.

▶ What you'll see: advantages have mean near zero while keeping the same deviations around expectation.

In [ ]:
grad_component_e3 = rng_e3.choice([-1.0, 1.0], size=200) # toy log-policy gradient signs.
updates_return_e3 = grad_component_e3 * returns_e3 # policy-gradient weights using raw returns.
updates_adv_e3 = grad_component_e3 * advs_e3 # policy-gradient weights using advantages.
print("update std with returns:", round(float(np.std(updates_return_e3)), 3)) # inspect noisier update scale.
print("update std with advantages:", round(float(np.std(updates_adv_e3)), 3)) # inspect centered update scale.

▶ What you'll see: advantage weighting reduces update scale caused by a large common baseline.

In [ ]:
plt.figure(figsize=(5, 3)) # create a histogram comparison.
plt.hist(updates_return_e3, bins=20, alpha=0.6, label="return weighted") # raw-return updates.
plt.hist(updates_adv_e3, bins=20, alpha=0.6, label="advantage weighted") # advantage updates.
plt.title("Easy 3: baseline centers actor updates") # title histogram.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: advantage-weighted updates are centered around zero instead of being dominated by the shared return level.

👀 Takeaway: the critic baseline reduces variance without needing to change which actions are truly better than expected.

### Easy 4 — Visualize bootstrapping bias from a bad critic

**Goal.** Show that a wrong next-state value creates a wrong target, because bootstrapping trades variance for bias. We build it in 3 steps.

In [ ]:
true_next_e4 = 2.0 # true payoff from state 1 in the toy environment.
estimates_e4 = np.array([0.0, 0.8, 1.5, 2.0, 2.5]) # possible critic estimates for V(next).
targets_e4 = 0.0 + 0.9 * estimates_e4 # bootstrap targets for investing from state 0.
true_target_e4 = 0.9 * true_next_e4 # correct one-step target if V(next) were exact.
print("targets:", np.round(targets_e4, 3)) # inspect target sensitivity to V(next).

▶ What you'll see: every next-value error is copied into the bootstrap target after discounting.

In [ ]:
bias_e4 = targets_e4 - true_target_e4 # target bias caused by critic error.
print("target bias:", np.round(bias_e4, 3)) # inspect under- and over-estimation.
assert round(true_target_e4, 3) == 1.800 # verify correct target for invest.

▶ What you'll see: underestimating V(next) underestimates the investment action.

In [ ]:
plt.figure(figsize=(5, 3)) # create a bias plot.
plt.plot(estimates_e4, targets_e4, marker="o", label="bootstrap target") # show target by estimate.
plt.axhline(true_target_e4, color="black", linestyle="--", label="true target") # show correct target.
plt.title("Easy 4: bootstrapping inherits critic error") # title the plot.
plt.xlabel("critic estimate V(next)") # label x-axis.
plt.ylabel("target") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the target line crosses the true target only when the next value is accurate.

👀 Takeaway: bootstrapping is efficient, but its target is only as trustworthy as the critic estimate it uses.

### Easy 5 — Add entropy pressure for exploration

**Goal.** Measure policy entropy, because exploration pressure discourages the actor from becoming deterministic too early. We build it in 3 steps.

In [ ]:
policies_e5 = np.array([[0.5, 0.5], [0.9, 0.1], [0.99, 0.01]]) # increasingly deterministic policies.
entropy_e5 = -np.sum(policies_e5 * np.log(policies_e5 + 1e-12), axis=1) # compute categorical entropy.
print("entropies:", np.round(entropy_e5, 3)) # inspect exploration amount.
assert entropy_e5[0] > entropy_e5[1] > entropy_e5[2] # verify entropy falls as policy sharpens.

▶ What you'll see: the uniform policy has the largest entropy.

In [ ]:
bonus_coef_e5 = 0.05 # small entropy coefficient.
advantage_e5 = 0.8 # example positive advantage.
objective_e5 = advantage_e5 + bonus_coef_e5 * entropy_e5 # toy objective with exploration bonus.
print("advantage plus entropy bonus:", np.round(objective_e5, 3)) # inspect how entropy changes objective.

▶ What you'll see: high-entropy policies receive a slightly larger exploration-augmented score.

In [ ]:
plt.figure(figsize=(4, 3)) # create an entropy plot.
plt.bar(["uniform", "peaked", "near-det"], entropy_e5, color="purple") # compare entropy by policy.
plt.title("Easy 5: entropy measures exploration") # title the plot.
plt.ylabel("entropy") # label entropy scale.
plt.xticks(rotation=15) # rotate labels for readability.
plt.show() # display the plot.

▶ What you'll see: entropy collapses as the actor puts almost all probability on one action.

👀 Takeaway: entropy bonuses keep policy support wide enough for the critic to keep receiving useful evidence.

## 🔴 Advanced

### Advanced 1 — Train synchronous A2C-style batches

**Goal.** Average several episode gradients before updating the actor, because synchronous actor-critic reduces noisy single-trajectory updates. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11) # reproducible batch sampling.
logits_a1 = np.zeros((2, 2)) # actor logits.
V_a1 = np.zeros(2) # critic values.
prob_hist_a1 = [] # track invest probability.
print("batch size:", 8) # inspect the synchronous batch size.

▶ What you'll see: the batch trainer starts from a uniform policy.

In [ ]:
for update_a1 in range(80): # perform multiple synchronous updates.
    grad_acc_a1 = np.zeros_like(logits_a1) # accumulate actor gradients.
    value_updates_a1 = np.zeros_like(V_a1) # accumulate critic TD errors.
    value_counts_a1 = np.zeros_like(V_a1) # count visits per state.
    for episode_a1 in range(8): # collect a small batch of episodes.
        state_a1 = 0 # reset.
        done_a1 = False # episode flag.
        while not done_a1: # rollout.
            probs_a1 = softmax(logits_a1[state_a1]) # current policy.
            action_a1 = int(rng_a1.choice(2, p=probs_a1)) # sample action.
            next_a1, reward_a1, done_a1 = toy_step(state_a1, action_a1) # transition.
            target_a1 = reward_a1 + (0.0 if done_a1 else 0.9 * V_a1[next_a1]) # TD target.
            delta_a1 = target_a1 - V_a1[state_a1] # advantage.
            grad_acc_a1[state_a1] += delta_a1 * grad_log_softmax(probs_a1, action_a1) # actor gradient.
            value_updates_a1[state_a1] += delta_a1 # critic correction.
            value_counts_a1[state_a1] += 1 # visit count.
            state_a1 = next_a1 # advance.
    logits_a1 += 0.08 * grad_acc_a1 / 8.0 # apply averaged actor update.
    mask_a1 = value_counts_a1 > 0 # states visited in the batch.
    V_a1[mask_a1] += 0.25 * value_updates_a1[mask_a1] / value_counts_a1[mask_a1] # averaged critic update.
    prob_hist_a1.append(softmax(logits_a1[0])[1]) # track policy.
print("final π(invest):", round(prob_hist_a1[-1], 3)) # inspect learned policy.
assert prob_hist_a1[-1] > 0.7 # verify learning occurred.

▶ What you'll see: batch updates still learn the delayed payoff action.

In [ ]:
print("final V:", np.round(V_a1, 3)) # inspect critic after batch training.
assert V_a1[0] > 1.0 # start state has learned positive value.

▶ What you'll see: the critic has learned useful values alongside the actor.

In [ ]:
plt.figure(figsize=(5, 3)) # create a policy curve.
plt.plot(prob_hist_a1, color="seagreen") # show batch actor improvement.
plt.title("Advanced 1: synchronous actor-critic updates") # title the plot.
plt.xlabel("batch update") # label update axis.
plt.ylabel("π(invest|s0)") # label policy probability.
plt.ylim(0, 1) # fixed probability scale.
plt.show() # display the plot.

▶ What you'll see: the curve is smoother than many single-transition updates because gradients are averaged.

👀 Takeaway: A2C-style averaging reduces variance by updating from a batch of actor-critic experience.

### Advanced 2 — Compare Monte Carlo and TD advantage variance

**Goal.** Compare full-return advantages with one-step TD advantages, because actor-critic chooses the TD estimate to lower variance. We build it in 3 steps.

In [ ]:
rng_a2 = np.random.default_rng(12) # reproducible noisy rewards.
noise_a2 = rng_a2.normal(0.0, 0.7, size=300) # terminal reward noise.
mc_returns_a2 = 0.0 + 0.9 * (2.0 + noise_a2) # full sampled return after investing.
critic_next_a2 = 2.0 # critic's expected next value.
td_adv_a2 = np.full_like(mc_returns_a2, 0.9 * critic_next_a2 - 1.0) # one-step TD advantage using baseline V(s0)=1.
mc_adv_a2 = mc_returns_a2 - 1.0 # Monte Carlo advantage using sampled full return.
print("MC advantage std:", round(float(np.std(mc_adv_a2)), 3)) # inspect sampled-return variance.
print("TD advantage std:", round(float(np.std(td_adv_a2)), 3)) # inspect bootstrapped variance.

▶ What you'll see: the TD advantage has much lower variance in this setup.

In [ ]:
assert np.std(td_adv_a2) < np.std(mc_adv_a2) # verify bootstrapping reduced variance here.
print("MC mean:", round(float(np.mean(mc_adv_a2)), 3), "TD value:", round(float(td_adv_a2[0]), 3)) # compare central tendency.

▶ What you'll see: TD is stable, while MC fluctuates around a similar consequence estimate.

In [ ]:
plt.figure(figsize=(5, 3)) # create histogram comparison.
plt.hist(mc_adv_a2, bins=25, alpha=0.65, label="Monte Carlo A") # full-return advantages.
plt.axvline(td_adv_a2[0], color="red", linewidth=2, label="TD A") # one-step TD advantage.
plt.title("Advanced 2: TD advantage lowers variance") # title the plot.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: MC advantages spread across a range, while TD is a single stable estimate.

👀 Takeaway: TD advantage is lower variance but can be biased if the critic is wrong.

### Advanced 3 — Show off-policy support failure

**Goal.** Demonstrate why rarely sampled actions are hard to evaluate, because policy updates cannot trust actions absent from the data. We build it in 3 steps.

In [ ]:
behavior_probs_a3 = np.array([0.98, 0.02]) # behavior policy almost never invests.
target_probs_a3 = np.array([0.2, 0.8]) # target policy wants to invest often.
rng_a3 = np.random.default_rng(13) # reproducible action samples.
actions_a3 = rng_a3.choice(2, size=500, p=behavior_probs_a3) # offline data actions.
counts_a3 = np.bincount(actions_a3, minlength=2) # count support for each action.
print("action counts:", counts_a3) # inspect data support.

▶ What you'll see: the investment action appears only a few times.

In [ ]:
importance_a3 = target_probs_a3[actions_a3] / behavior_probs_a3[actions_a3] # importance weights.
print("max importance weight:", round(float(np.max(importance_a3)), 1)) # inspect weight explosion for rare target action.
print("mean importance weight:", round(float(np.mean(importance_a3)), 3)) # inspect average correction.
assert np.max(importance_a3) > 10 # rare actions get large weights.

▶ What you'll see: rare behavior-policy actions receive large target/behavior ratios.

In [ ]:
plt.figure(figsize=(5, 3)) # create support plot.
plt.bar(["safe a0", "invest a1"], counts_a3, color=["gray", "crimson"]) # show action support.
plt.title("Advanced 3: off-policy support problem") # title the plot.
plt.ylabel("count in data") # label data-count axis.
plt.show() # display the plot.

▶ What you'll see: the action the target policy wants most is barely represented in the data.

👀 Takeaway: actor-critic estimates are unreliable when the action being improved was rarely sampled.

### Advanced 4 — Sweep critic learning rate stability

**Goal.** Compare critic step sizes, because bootstrapping from a moving target can amplify error when updates are too aggressive. We build it in 3 steps.

In [ ]:
alphas_a4 = np.array([0.05, 0.2, 0.8, 1.2]) # critic learning rates to compare.
curves_a4 = [] # store value trajectories.
target_a4 = 1.72 # fixed target for this diagnostic.
print("critic learning rates:", alphas_a4) # inspect sweep.

▶ What you'll see: the sweep includes cautious and overly aggressive rates.

In [ ]:
for alpha_a4 in alphas_a4: # simulate repeated updates toward a fixed target.
    v_a4 = 0.4 # start from old value.
    curve_a4 = [] # current rate trajectory.
    for step_a4 in range(20): # repeated critic corrections.
        v_a4 = v_a4 + alpha_a4 * (target_a4 - v_a4) # TD-style value update.
        curve_a4.append(v_a4) # store trajectory.
    curves_a4.append(curve_a4) # save this alpha's curve.
print("final values:", [round(c[-1], 3) for c in curves_a4]) # inspect convergence or oscillation.

▶ What you'll see: moderate rates approach the target smoothly; very large rates can overshoot.

In [ ]:
plt.figure(figsize=(5, 3)) # create trajectory plot.
for alpha_a4, curve_a4 in zip(alphas_a4, curves_a4): # plot each learning rate.
    plt.plot(curve_a4, label=f"α={alpha_a4}") # add curve.
plt.axhline(target_a4, color="black", linestyle="--", label="target") # show target.
plt.title("Advanced 4: critic step-size stability") # title the plot.
plt.xlabel("update") # label update axis.
plt.ylabel("V estimate") # label value scale.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: step size controls whether the critic glides to the target or oscillates around it.

👀 Takeaway: bootstrapped critics need conservative enough steps to avoid chasing their own errors.

### Advanced 5 — Add UCB-style exploration diagnostics

**Goal.** Compute an uncertainty bonus, because exploration methods deliberately value actions whose estimates are uncertain. We build it in 3 steps.

In [ ]:
means_a5 = np.array([0.55, 0.70]) # current empirical action-value means.
counts_a5 = np.array([5, 40]) # action 0 has much less evidence.
t_a5 = 20 # current time index from the lesson calculation.
c_a5 = 1.0 # exploration strength.
bonus_a5 = c_a5 * np.sqrt(2 * np.log(t_a5) / counts_a5) # UCB uncertainty bonus.
print("bonuses:", np.round(bonus_a5, 3)) # inspect exploration pressure.

▶ What you'll see: the less-sampled action receives the larger uncertainty bonus.

In [ ]:
ucb_a5 = means_a5 + bonus_a5 # optimistic action scores.
print("UCB scores:", np.round(ucb_a5, 3)) # inspect mean plus uncertainty.
assert round(float(ucb_a5[0]), 3) == 1.645 # verify concrete lesson number for count 5.

▶ What you'll see: action 0's score is much higher than its observed mean because evidence is scarce.

In [ ]:
x_a5 = np.arange(2) # x positions for grouped bars.
plt.figure(figsize=(5, 3)) # create UCB decomposition plot.
plt.bar(x_a5 - 0.18, means_a5, width=0.36, label="mean", color="gray") # empirical mean.
plt.bar(x_a5 + 0.18, ucb_a5, width=0.36, label="mean + bonus", color="seagreen") # optimistic score.
plt.xticks(x_a5, ["a0 few samples", "a1 many samples"]) # label actions.
plt.title("Advanced 5: exploration bonus") # title the plot.
plt.ylabel("score") # label score axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: uncertainty can make a lower-mean action worth trying again.

👀 Takeaway: exploration pressure protects actor-critic from prematurely trusting a narrow slice of sampled actions.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Actor-critic lowers policy-gradient variance by learning a baseline beside the policy.

The actor chooses actions while the critic estimates value. Bootstrapped TD errors become advantages, reducing variance without leaving the policy-gradient family. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1113
rng = np.random.default_rng(SEED)
random.seed(SEED)

ACTIONS = np.array([
    [-1, 0],
    [1, 0],
    [0, -1],
    [0, 1],
])
ACTION_NAMES = np.array(["U", "D", "L", "R"])

@dataclass
class LadderEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    walls: tuple
    slip: float
    wind: float
    step_cost: float
    goal_reward: float
    traps: dict
    max_steps: int
    continuous: bool = False

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return len(ACTIONS)

    def state_index(self, pos):
        row, col = pos
        return row * self.width + col

    def index_state(self, idx):
        row = idx // self.width
        col = idx % self.width
        return (row, col)

    def reset(self):
        return self.state_index(self.start)

    def move(self, pos, action, local_rng):
        actual = int(action)
        if local_rng.random() < self.slip:
            actual = int(local_rng.integers(0, self.n_actions))
        delta = ACTIONS[actual].copy()
        if self.wind > 0.0 and local_rng.random() < self.wind:
            delta = delta + np.array([-1, 0])
        nxt = (pos[0] + int(delta[0]), pos[1] + int(delta[1]))
        bad_row = nxt[0] < 0 or nxt[0] >= self.height
        bad_col = nxt[1] < 0 or nxt[1] >= self.width
        if bad_row or bad_col or nxt in self.walls:
            nxt = pos
        return nxt

    def step(self, state, action, local_rng):
        pos = self.index_state(int(state))
        nxt = self.move(pos, action, local_rng)
        reward = self.step_cost
        done = False
        if nxt in self.traps:
            reward = reward + float(self.traps[nxt])
        if nxt == self.goal:
            reward = reward + self.goal_reward
            done = True
        return self.state_index(nxt), reward, done


def make_rl_ladder(continuous=False):
    envs = []
    envs.append(LadderEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), tuple(), 0.0, 0.0, 0.0, 1.0, {}, 4, continuous))
    envs.append(LadderEnv("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), tuple(), 0.15, 0.0, -0.01, 1.0, {}, 8, continuous))
    envs.append(LadderEnv("D3 4x4 gridworld", 4, 4, (3, 0), (0, 3), ((1, 1),), 0.05, 0.0, -0.02, 1.0, {(2, 2): -0.25}, 24, continuous))
    envs.append(LadderEnv("D4 windy stochastic grid", 5, 5, (4, 0), (0, 4), ((1, 1), (2, 1), (3, 3)), 0.12, 0.18, -0.025, 1.1, {(2, 3): -0.4}, 35, continuous))
    envs.append(LadderEnv("D5 sparse reward grid", 6, 6, (5, 0), (0, 5), ((1, 1), (1, 2), (2, 2), (3, 4), (4, 1)), 0.18, 0.20, -0.03, 1.5, {(2, 4): -0.6, (4, 4): -0.3}, 50, continuous))
    return envs


def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)


def discounted_returns(rewards, gamma):
    returns = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = float(rewards[t]) + gamma * running
        returns[t] = running
    return returns


def rollout(env, logits, local_rng, gamma=0.9):
    states = []
    actions = []
    rewards = []
    state = env.reset()
    for _ in range(env.max_steps):
        probs = softmax(logits[state])
        action = int(local_rng.choice(env.n_actions, p=probs))
        next_state, reward, done = env.step(state, action, local_rng)
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        state = next_state
        if done:
            break
    returns = discounted_returns(np.array(rewards), gamma)
    return np.array(states), np.array(actions), np.array(rewards), returns


def evaluate_policy(env, logits, episodes=20, gamma=0.9):
    values = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        _, _, rewards, _ = rollout(env, logits, local_rng, gamma)
        values.append(float(np.sum(rewards)))
    return float(np.mean(values))


def value_iteration(env, gamma=0.9, iterations=120):
    values = np.zeros(env.n_states)
    local_rng = np.random.default_rng(SEED)
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            pos = env.index_state(state)
            if pos == env.goal:
                continue
            q_values = []
            for action in range(env.n_actions):
                next_state, reward, done = env.step(state, action, local_rng)
                q_values.append(reward + gamma * values[next_state] * (1.0 - float(done)))
            new_values[state] = np.max(q_values)
        values = new_values
    return values


def greedy_logits_from_values(env, values, gamma=0.9, scale=5.0):
    logits = np.zeros((env.n_states, env.n_actions))
    local_rng = np.random.default_rng(SEED + 7)
    for state in range(env.n_states):
        for action in range(env.n_actions):
            next_state, reward, done = env.step(state, action, local_rng)
            logits[state, action] = scale * (reward + gamma * values[next_state] * (1.0 - float(done)))
    return logits


def plot_policy_panel(ax, env, values, logits, title):
    grid = values.reshape(env.height, env.width)
    ax.imshow(grid, cmap="viridis")
    probs = softmax(logits)
    for state in range(env.n_states):
        row, col = env.index_state(state)
        if (row, col) in env.walls:
            ax.text(col, row, "#", ha="center", va="center", color="white")
            continue
        best = int(np.argmax(probs[state]))
        ax.text(col, row, ACTION_NAMES[best], ha="center", va="center", color="white")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(envs):
    for env in envs:
        print(env.name, "states", env.n_states, "actions", env.n_actions, "slip", env.slip, "wind", env.wind)
        print("start", env.start, "goal", env.goal, "walls", len(env.walls), "traps", env.traps)


def train_reinforce(env, episodes=80, gamma=0.9, lr=0.08, baseline=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        advantages = returns.copy()
        if baseline:
            advantages = returns - value[states]
            for state, target in zip(states, returns):
                value[state] = value[state] + 0.15 * (target - value[state])
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def train_actor_critic(env, episodes=80, gamma=0.9, actor_lr=0.05, critic_lr=0.12, normalize=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 2 * env.n_states)
    for _ in range(episodes):
        states = []
        actions = []
        deltas = []
        rewards = []
        state = env.reset()
        for _ in range(env.max_steps):
            probs = softmax(logits[state])
            action = int(local_rng.choice(env.n_actions, p=probs))
            next_state, reward, done = env.step(state, action, local_rng)
            target = reward + gamma * value[next_state] * (1.0 - float(done))
            delta = target - value[state]
            value[state] = value[state] + critic_lr * delta
            states.append(state)
            actions.append(action)
            deltas.append(delta)
            rewards.append(reward)
            state = next_state
            if done:
                break
        advantages = np.array(deltas)
        if normalize and len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + actor_lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def gae_advantages(rewards, values, next_values, dones, gamma=0.9, lam=0.95):
    deltas = rewards + gamma * next_values * (1.0 - dones) - values
    adv = np.zeros_like(rewards, dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return deltas, adv


def train_gae_policy(env, lam=0.95, episodes=80, gamma=0.9):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + int(100 * lam) + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        next_values = np.zeros(len(states))
        dones = np.zeros(len(states))
        for i, state in enumerate(states):
            if i + 1 < len(states):
                next_values[i] = value[states[i + 1]]
            else:
                dones[i] = 1.0
        _, advantages = gae_advantages(rewards, value[states], next_values, dones, gamma, lam)
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.12 * (target - value[state])
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + 0.05 * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def ppo_surrogate(old_probs, new_probs, actions, advantages, clip=0.2):
    chosen_old = old_probs[np.arange(len(actions)), actions]
    chosen_new = new_probs[np.arange(len(actions)), actions]
    ratios = chosen_new / np.maximum(chosen_old, 1e-8)
    unclipped = ratios * advantages
    clipped = np.clip(ratios, 1.0 - clip, 1.0 + clip) * advantages
    return ratios, np.minimum(unclipped, clipped)


def train_ppo(env, episodes=80, gamma=0.9, clip=0.2, clipped=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 3 * env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        old_probs = softmax(logits[states])
        advantages = returns - value[states]
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for _epoch in range(3):
            new_probs = softmax(logits[states])
            ratios, weights = ppo_surrogate(old_probs, new_probs, actions, advantages, clip)
            if not clipped:
                weights = ratios * advantages
            for state, action, weight in zip(states, actions, weights):
                probs = softmax(logits[state])
                grad = -probs
                grad[action] = grad[action] + 1.0
                logits[state] = logits[state] + 0.04 * weight * grad
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.15 * (target - value[state])
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def distributional_backup(rewards, gamma):
    atoms = discounted_returns(np.array(rewards, dtype=float), gamma)
    probs = np.ones_like(atoms) / len(atoms)
    return atoms, probs, float(atoms[0])


def train_distributional(env, atoms=21, episodes=60, gamma=0.9):
    support = np.linspace(-1.0, 2.0, atoms)
    pmf = np.ones((env.n_states, env.n_actions, atoms)) / atoms
    logits = np.zeros((env.n_states, env.n_actions))
    curve = []
    errors = []
    optimal = value_iteration(env, gamma)
    local_rng = np.random.default_rng(SEED + 4 * env.n_states)
    for _ in range(episodes):
        state = env.reset()
        episode_reward = 0.0
        for _step in range(env.max_steps):
            means = np.sum(pmf[state] * support[None, :], axis=1)
            action = int(np.argmax(means + local_rng.normal(0.0, 0.03, env.n_actions)))
            next_state, reward, done = env.step(state, action, local_rng)
            next_action = int(np.argmax(np.sum(pmf[next_state] * support[None, :], axis=1)))
            shifted = reward + gamma * support * (1.0 - float(done))
            target = np.interp(support, shifted, pmf[next_state, next_action], left=0.0, right=0.0)
            if np.sum(target) <= 0.0:
                nearest = int(np.argmin(np.abs(support - reward)))
                target = np.zeros(atoms)
                target[nearest] = 1.0
            target = target / np.sum(target)
            pmf[state, action] = 0.9 * pmf[state, action] + 0.1 * target
            episode_reward = episode_reward + reward
            state = next_state
            if done:
                break
        learned = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
        errors.append(float(np.mean(np.abs(learned - optimal))))
        curve.append(episode_reward)
    values = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
    logits = greedy_logits_from_values(env, values, gamma)
    spread = np.mean(np.std(pmf * support[None, None, :], axis=2))
    return logits, values, np.array(errors), float(spread)


def continuous_features(state, action):
    return np.array([1.0, state, action, state * action, action * action])


def deterministic_actor_critic(reward, next_q, gamma):
    target = reward + gamma * next_q
    critic_prediction = 0.4
    critic_error = target - critic_prediction
    return target, critic_error


def td3_toy_update(state, reward, next_state, actor_w, q1_w, q2_w, gamma=0.9, noise=0.05):
    action = float(np.tanh(actor_w * state))
    next_action = float(np.clip(np.tanh(actor_w * next_state) + noise, -1.0, 1.0))
    q1_next = float(continuous_features(next_state, next_action) @ q1_w)
    q2_next = float(continuous_features(next_state, next_action) @ q2_w)
    target = reward + gamma * min(q1_next, q2_next)
    prediction = float(continuous_features(state, action) @ q1_w)
    td_error = target - prediction
    q1_w = q1_w + 0.05 * td_error * continuous_features(state, action)
    return action, target, td_error, q1_w


def train_td3_ladder(env, episodes=70, gamma=0.9, twin=True):
    actor_w = 0.2
    q1_w = np.array([0.0, 0.2, 0.1, 0.0, -0.05])
    q2_w = np.array([-0.02, 0.15, 0.08, 0.0, -0.08])
    curve = []
    local_rng = np.random.default_rng(SEED + 5 * env.n_states)
    for _ in range(episodes):
        state = 0.0
        total = 0.0
        for _step in range(env.max_steps):
            action = float(np.clip(np.tanh(actor_w * state) + local_rng.normal(0.0, 0.15), -1.0, 1.0))
            target_position = 1.0
            next_state = float(np.clip(state + 0.25 * action + local_rng.normal(0.0, 0.03), -1.2, 1.2))
            reward = -abs(target_position - next_state) - 0.05 * action * action
            next_action = float(np.clip(np.tanh(actor_w * next_state) + local_rng.normal(0.0, 0.05), -1.0, 1.0))
            q1_next = float(continuous_features(next_state, next_action) @ q1_w)
            q2_next = float(continuous_features(next_state, next_action) @ q2_w)
            next_value = min(q1_next, q2_next) if twin else q1_next
            target = reward + gamma * next_value
            feat = continuous_features(state, action)
            td_error_1 = target - float(feat @ q1_w)
            td_error_2 = target - float(feat @ q2_w)
            q1_w = q1_w + 0.03 * td_error_1 * feat
            q2_w = q2_w + 0.03 * td_error_2 * feat
            actor_grad = q1_w[2] + q1_w[3] * state + 2.0 * q1_w[4] * np.tanh(actor_w * state)
            actor_w = actor_w + 0.01 * actor_grad * state
            total = total + reward
            state = next_state
        curve.append(total)
    values = np.linspace(-1.0, 1.0, env.n_states)
    logits = np.tile(np.array([-actor_w, actor_w, -0.5 * actor_w, 0.5 * actor_w]), (env.n_states, 1))
    return logits, values, np.array(curve)


## The concept, built once: actor plus critic
Actor-critic still uses $$
abla_\theta J(\theta)=\mathbb{E}[
abla_\theta\log\pi_\theta(a_t\mid s_t)A_t]$$, but estimates $A_t$ with a critic. The lesson one-step target is $y=1+0.9\cdot0.8=1.720$, and with $Q_{old}=0.4$, $\alpha=0.5$ gives $Q_{new}=1.060$.

In [ ]:

def actor_critic_step(logits, value, action, reward, next_value, gamma=0.9, actor_lr=0.1, critic_lr=0.5):
    target = reward + gamma * next_value
    delta = target - value
    new_value = value + critic_lr * delta
    probs = softmax(logits)
    grad = -probs
    grad[action] = grad[action] + 1.0
    new_logits = logits + actor_lr * delta * grad
    return target, delta, new_value, new_logits

logits = np.array([1.0, 0.0])
target, delta, new_value, new_logits = actor_critic_step(logits, 0.4, 0, 1.0, 0.8)

print("TD target", target)
print("TD error advantage", delta)
print("critic update", new_value)
print("actor logits", new_logits)

assert round(target, 3) == 1.720
assert round(delta, 3) == 1.320
assert round(new_value, 3) == 1.060


The critic's TD error is a low-variance advantage estimate. The actor receives only the direction from $
abla\log\pi$, while the critic absorbs most of the value-scale learning.

In [ ]:

probs_before = softmax(logits)
probs_after = softmax(new_logits)

print("before", probs_before)
print("after", probs_after)
assert probs_after[0] > probs_before[0]


## The dataset ladder: F12 sequential-decision environments
We build the D1-D5 ladder inline, from a two-state chain to a sparse stochastic windy grid. Every rung has a start, goal, transition noise, and a small enough state space for CPU-only NumPy experiments.

In [ ]:

envs = make_rl_ladder()
summarize_ladder(envs)

for env in envs:
    sample_state = env.reset()
    sample_next, sample_reward, sample_done = env.step(sample_state, 3, np.random.default_rng(SEED))
    print(env.name, "sample", sample_state, "->", sample_next, "reward", round(sample_reward, 3), "done", sample_done)


## Run synchronous actor-critic across D1-D5
This is A2C-style synchronous tabular actor-critic, not multiprocessing A3C. The metric is average return.

In [ ]:

results = []
artifacts = []
for env in envs:
    logits, values, curve = train_actor_critic(env, normalize=True)
    learned_return = evaluate_policy(env, logits)
    artifacts.append((env, logits, values, curve))
    results.append((env.name, learned_return, float(np.mean(curve[-10:]))))

print("rung | evaluated_return | last10_training_return")
for name, learned_return, recent_return in results:
    print(name, round(learned_return, 3), round(recent_return, 3))


## Results visualization
Policy/value panels show the actor and critic together; the lower row tracks return.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, (env, logits, values, curve) in zip(axes[0], artifacts):
    plot_policy_panel(ax, env, values, logits, env.name)
for ax, (env, logits, values, curve) in zip(axes[1], artifacts):
    ax.plot(curve, color="tab:orange")
    ax.set_title("return " + env.name.split()[0])
    ax.set_xlabel("episode")
    ax.set_ylabel("return")
plt.tight_layout()


## Pitfall on D5: bootstrapping from a moving target
A critic step that is too large can chase its own target. The fix uses a smaller critic step plus advantage normalization.

In [ ]:

d5 = envs[-1]
unstable_logits, unstable_values, unstable_curve = train_actor_critic(d5, critic_lr=0.9, normalize=False)
stable_logits, stable_values, stable_curve = train_actor_critic(d5, critic_lr=0.12, normalize=True)
unstable_scale = float(np.max(np.abs(unstable_values)))
stable_scale = float(np.max(np.abs(stable_values)))

print("unstable critic scale", round(unstable_scale, 3))
print("stable critic scale", round(stable_scale, 3))
print("stable evaluated return", round(evaluate_policy(d5, stable_logits), 3))
assert stable_scale <= unstable_scale + 10.0


## Evaluate it + practice
- Metric: compare return or advantage/value error against a no-skill random-policy baseline on every rung.
- Sanity check: D1 should solve first because it has the shortest horizon and no stochastic transition.
- Ablation: turn off the key stabilizer for this lesson and the D5 metric should drop or become noisier.
- Failure signals: exploding logits, a value table with impossible magnitudes, or a policy that never reaches the goal.

Practice prompts:
1. Change $\gamma$ from 0.9 to 0.7 and explain which rungs lose the most return.
2. Set critic_lr to 0 and show the actor loses its advantage signal.
3. Compare normalized and raw TD errors on D4.


In [ ]:
# Your code here


In [ ]:
# Your code here
